In [1]:
!nvidia-smi

Sun May  3 08:41:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [24]:
from google.colab import files
uploaded = files.upload()

Saving milestone3_complete.cu to milestone3_complete (6).cu


In [12]:
# Verify CUDA
!nvcc --version

# Copy all datasets from Drive
import shutil
import os

# Your datasets are in PDC_datasets folder inside MyDrive
drive_base = "/content/drive/MyDrive/PDC_datasets/"

datasets = [
    "Iris.csv",
    "shuttle.csv",
    "letter-recognition.csv",
    "Skin_NonSkin.csv",
    "synthetic_1000k_200f.csv"
]

print("Copying datasets from Drive...")
for ds in datasets:
    src = drive_base + ds
    if os.path.exists(src):
        # For large file (1.68 GB), copy might take time
        print(f"  Copying {ds}... (size: {os.path.getsize(src) / (1024**3):.2f} GB)" if 'synthetic' in ds else f"  Copying {ds}...")
        shutil.copy2(src, ds)
        print(f"  ✓ Copied {ds}")
    else:
        print(f"  ✗ Not found: {src}")

# List files to verify
print("\nFiles in current directory:")
!ls -lh *.csv

/bin/bash: line 1: nvcc: command not found
Copying datasets from Drive...
  Copying Iris.csv...
  ✓ Copied Iris.csv
  Copying shuttle.csv...
  ✓ Copied shuttle.csv
  Copying letter-recognition.csv...
  ✓ Copied letter-recognition.csv
  Copying Skin_NonSkin.csv...
  ✓ Copied Skin_NonSkin.csv
  Copying synthetic_1000k_200f.csv... (size: 1.68 GB)
  ✓ Copied synthetic_1000k_200f.csv

Files in current directory:
-rw-r--r-- 1 root root  142 May  3 16:04 benchmark_forest_train_m3_cpu.csv
-rw-r--r-- 1 root root 1.2K May  3 14:04 benchmark_scalability_synthetic_converted.csv
-rw-r--r-- 1 root root 1.2K May  3 14:04 benchmark_scalability_synthetic_only_m3.csv
-rw-r--r-- 1 root root  828 May  3 16:04 faraz_forest.csv
-rw-r--r-- 1 root root  828 May  3 16:04 faraz_speed.csv
-rw-r--r-- 1 root root  828 May  3 16:04 faraz_test.csv
-rw------- 1 root root 3.2K Mar 28 09:35 Iris.csv
-rw------- 1 root root 729K Mar 27 17:05 letter-recognition.csv
-rw-r--r-- 1 root root  828 May  3 16:04 m3_cap.csv
-rw--

In [25]:
!nvcc -O2 -std=c++17 -arch=sm_75 "milestone3_complete (6).cu" -o milestone3


In [27]:
!./milestone3

   MILESTONE 3 — Random Forest + Parallel Inference
   Hardware threads: 2

=== SANITY TESTS ===

[Test: bootstrap_sample with replacement]
  unique indices drawn: 63/100  [PASS]

[Test: tree count capped at MAX_TREES=10]
  [Sequential] Tree 1/10  nodes=9  gpu_split_ms=0.55  total_ms=307.41
  [Sequential] Tree 2/10  nodes=9  gpu_split_ms=0.32  total_ms=0.62
  [Sequential] Tree 3/10  nodes=9  gpu_split_ms=0.33  total_ms=0.61
  [Sequential] Tree 4/10  nodes=9  gpu_split_ms=0.30  total_ms=0.61
  [Sequential] Tree 5/10  nodes=9  gpu_split_ms=0.31  total_ms=0.59
  [Sequential] Tree 6/10  nodes=9  gpu_split_ms=0.73  total_ms=1.02
  [Sequential] Tree 7/10  nodes=9  gpu_split_ms=0.32  total_ms=0.61
  [Sequential] Tree 8/10  nodes=9  gpu_split_ms=0.30  total_ms=0.58
  [Sequential] Tree 9/10  nodes=9  gpu_split_ms=0.30  total_ms=0.60
  [Sequential] Tree 10/10  nodes=9  gpu_split_ms=0.31  total_ms=0.59
  forest size=10  [PASS]

[Test: compact tree serialization matches pointer tree]
  all 18 test

In [28]:
import pandas as pd

print("="*60)
print("CONVERTING M3 CSVs TO SECONDS WITH CONSISTENT NAMING")
print("="*60)

# 1. FOREST TRAIN CSV (matches M2's train_time_sec)
print("\n1. Converting benchmark_forest_train_m3.csv...")
df_train = pd.read_csv('benchmark_forest_train_m3.csv')
df_train['train_time_sec'] = df_train['wall_clock_ms'] / 1000.0
df_train['total_tree_time_sec'] = df_train['sum_tree_ms'] / 1000.0
df_train['gpu_kernel_time_sec'] = df_train['gpu_kernel_ms'] / 1000.0
df_train['avg_tree_time_sec'] = df_train['avg_tree_ms'] / 1000.0
# Keep original columns for reference
df_train = df_train.drop(columns=['wall_clock_ms', 'sum_tree_ms', 'gpu_kernel_ms', 'avg_tree_ms'])
df_train.to_csv('benchmark_forest_train_m3_sec.csv', index=False)
print("   ✓ Saved benchmark_forest_train_m3_sec.csv")
print(f"   Columns: {list(df_train.columns)}")

# 2. SCALABILITY CSV (matches M2's train_time_sec)
print("\n2. Converting benchmark_scalability_m3.csv...")
df_scal = pd.read_csv('benchmark_scalability_m3.csv')
df_scal['train_time_sec'] = df_scal['wall_clock_ms'] / 1000.0
df_scal['gpu_time_sec'] = df_scal['gpu_kernel_ms'] / 1000.0
df_scal = df_scal.drop(columns=['wall_clock_ms', 'gpu_kernel_ms'])
df_scal.to_csv('benchmark_scalability_m3_sec.csv', index=False)
print("   ✓ Saved benchmark_scalability_m3_sec.csv")
print(f"   Columns: {list(df_scal.columns)}")

# 3. THROUGHPUT CSV (matches M2's predict_time_sec)
print("\n3. Converting benchmark_throughput_m3.csv...")
df_thru = pd.read_csv('benchmark_throughput_m3.csv')
df_thru['predict_time_sec'] = df_thru['elapsed_ms'] / 1000.0
df_thru = df_thru.drop(columns=['elapsed_ms'])
# Reorder columns to match M2 style
df_thru = df_thru[['variant', 'n_samples', 'n_trees', 'n_threads',
                   'predict_time_sec', 'samples_per_sec', 'speedup']]
df_thru.to_csv('benchmark_throughput_m3_sec.csv', index=False)
print("   ✓ Saved benchmark_throughput_m3_sec.csv")
print(f"   Columns: {list(df_thru.columns)}")

# 4. SPEEDUP CSV (M3-specific, but convert to sec)
print("\n4. Converting benchmark_speedup_vs_trees_m3.csv...")
df_speed = pd.read_csv('benchmark_speedup_vs_trees_m3.csv')
# This file has multiple datasets stacked (no dataset column - need to add)
# First, identify dataset boundaries based on n_trees resetting
datasets = ['Iris', 'Shuttle', 'LetterRecognition', 'Skin_NonSkin', 'Synthetic_1M_200f']
dataset_indices = []
current_dataset = 0
for i in range(len(df_speed)):
    if i > 0 and df_speed.iloc[i]['n_trees'] == 1 and df_speed.iloc[i-1]['n_trees'] == 10:
        current_dataset += 1
    dataset_indices.append(current_dataset)

df_speed['dataset'] = [datasets[i] for i in dataset_indices]
df_speed['seq_time_sec'] = df_speed['seq_ms'] / 1000.0
df_speed['parallel_time_sec'] = df_speed['par_sample_ms'] / 1000.0
df_speed = df_speed.drop(columns=['seq_ms', 'par_sample_ms'])
# Reorder
df_speed = df_speed[['dataset', 'n_trees', 'seq_time_sec', 'parallel_time_sec', 'speedup', 'throughput_sps']]
df_speed.to_csv('benchmark_speedup_vs_trees_m3_sec.csv', index=False)
print("   ✓ Saved benchmark_speedup_vs_trees_m3_sec.csv")
print(f"   Columns: {list(df_speed.columns)}")

print("\n" + "="*60)
print("✅ CONVERSION COMPLETE!")
print("="*60)
print("\nFiles created:")
print("  - benchmark_forest_train_m3_sec.csv")
print("  - benchmark_scalability_m3_sec.csv")
print("  - benchmark_throughput_m3_sec.csv")
print("  - benchmark_speedup_vs_trees_m3_sec.csv")

CONVERTING M3 CSVs TO SECONDS WITH CONSISTENT NAMING

1. Converting benchmark_forest_train_m3.csv...
   ✓ Saved benchmark_forest_train_m3_sec.csv
   Columns: ['dataset', 'variant', 'n_trees', 'n_samples', 'n_features', 'n_classes', 'test_accuracy', 'train_time_sec', 'total_tree_time_sec', 'gpu_kernel_time_sec', 'avg_tree_time_sec']

2. Converting benchmark_scalability_m3.csv...
   ✓ Saved benchmark_scalability_m3_sec.csv
   Columns: ['dataset', 'variant', 'fraction', 'n_samples', 'n_trees', 'test_accuracy', 'train_time_sec', 'gpu_time_sec']

3. Converting benchmark_throughput_m3.csv...
   ✓ Saved benchmark_throughput_m3_sec.csv
   Columns: ['variant', 'n_samples', 'n_trees', 'n_threads', 'predict_time_sec', 'samples_per_sec', 'speedup']

4. Converting benchmark_speedup_vs_trees_m3.csv...
   ✓ Saved benchmark_speedup_vs_trees_m3_sec.csv
   Columns: ['dataset', 'n_trees', 'seq_time_sec', 'parallel_time_sec', 'speedup', 'throughput_sps']

✅ CONVERSION COMPLETE!

Files created:
  - benchma

In [30]:
import os
from google.colab import files

# Download only the consistent (converted) files
consistent_files = [
    "benchmark_forest_train_m3_sec.csv",
    "benchmark_scalability_m3_sec.csv",
    "benchmark_throughput_m3_sec.csv",
    "benchmark_speedup_vs_trees_m3_sec.csv"
]

print("Downloading consistent M3 files...")
for file in consistent_files:
    if os.path.exists(file):
        files.download(file)
        print(f"  ✓ Downloaded {file}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  ✓ Downloaded benchmark_forest_train_m3_sec.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

  ✓ Downloaded benchmark_throughput_m3_sec.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  ✓ Downloaded benchmark_speedup_vs_trees_m3_sec.csv


In [31]:
from google.colab import files
uploaded = files.upload()

Saving synthetic_scalablity_m3.cu to synthetic_scalablity_m3.cu


In [35]:
!nvcc -O2 -std=c++17 -arch=sm_75 synthetic_scalablity_m3.cu -o synthetic_scale

synthetic_scalablity_m3.cu(173): warning #177-D: function "resolve_path" was declared but never referenced
  static std::string resolve_path(const std::string& f,
                     ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

synthetic_scalablity_m3.cu(1116): warning #177-D: function "measure_throughput" was declared but never referenced
  static std::vector<ThroughputResult> measure_throughput(
                                       ^

synthetic_scalablity_m3.cu(1145): warning #177-D: function "speedup_vs_trees" was declared but never referenced
  static std::vector<SpeedupRow> speedup_vs_trees(
                                 ^

synthetic_scalablity_m3.cu(1206): warning #177-D: function "run_variant_comparison" was declared but never referenced
  static void run_variant_comparison(
              ^

synthetic_scalablity_m3.cu(1282): warning #177-D: function "print_throughput_table" was declared but never referenced
  static void print_throughpu

In [36]:
!./synthetic_scale

   SYNTHETIC SCALABILITY - M3
  samples=1000000  features=200  classes=10

[Running Sequential variant...]
  [Sequential] Tree 1/10  nodes=233  gpu_split_ms=69.53  total_ms=797.80
  [Sequential] Tree 2/10  nodes=243  gpu_split_ms=68.47  total_ms=496.69
  [Sequential] Tree 3/10  nodes=237  gpu_split_ms=68.17  total_ms=492.87
  [Sequential] Tree 4/10  nodes=237  gpu_split_ms=68.29  total_ms=497.28
  [Sequential] Tree 5/10  nodes=249  gpu_split_ms=83.38  total_ms=508.28
  [Sequential] Tree 6/10  nodes=247  gpu_split_ms=82.46  total_ms=514.35
  [Sequential] Tree 7/10  nodes=239  gpu_split_ms=82.11  total_ms=503.87
  [Sequential] Tree 8/10  nodes=239  gpu_split_ms=75.58  total_ms=505.73
  [Sequential] Tree 9/10  nodes=235  gpu_split_ms=75.76  total_ms=499.25
  [Sequential] Tree 10/10  nodes=249  gpu_split_ms=73.15  total_ms=505.53
    [Scalability] Sequential frac=0.10 n=100000 wall_ms=5333.9
  [Sequential] Tree 1/10  nodes=245  gpu_split_ms=258.24  total_ms=1669.02
  [Sequential] Tree 2/10

In [2]:
from google.colab import files
uploaded = files.upload()

Saving benchmark_scalability_synthetic_only_m3.csv to benchmark_scalability_synthetic_only_m3.csv


In [4]:
import pandas as pd

# Load synthetic file
df_synth = pd.read_csv('benchmark_scalability_synthetic_only_m3.csv')

# Convert ms to seconds and rename columns to match main file
df_synth_converted = df_synth.copy()
df_synth_converted['train_time_sec'] = df_synth_converted['wall_clock_ms'] / 1000.0
df_synth_converted['gpu_time_sec'] = df_synth_converted['gpu_kernel_ms'] / 1000.0

# Drop the old ms columns
df_synth_converted = df_synth_converted.drop(columns=['wall_clock_ms', 'gpu_kernel_ms'])

# Reorder columns to match main file
df_synth_converted = df_synth_converted[['dataset', 'variant', 'fraction', 'n_samples',
                                          'n_trees', 'test_accuracy', 'train_time_sec', 'gpu_time_sec']]

# Save converted file
df_synth_converted.to_csv('benchmark_scalability_synthetic_converted.csv', index=False)

print("Converted file saved: benchmark_scalability_synthetic_converted.csv")
print("\nPreview of converted data:")
print(df_synth_converted.head())

Converted file saved: benchmark_scalability_synthetic_converted.csv

Preview of converted data:
             dataset     variant  fraction  n_samples  n_trees  test_accuracy  \
0  Synthetic_1M_200f  Sequential      0.10     100000       10         0.9906   
1  Synthetic_1M_200f  Sequential      0.25     250000       10         0.9943   
2  Synthetic_1M_200f  Sequential      0.50     500000       10         0.9928   
3  Synthetic_1M_200f  Sequential      0.75     750000       10         0.9875   
4  Synthetic_1M_200f  Sequential      1.00    1000000       10         0.9878   

   train_time_sec  gpu_time_sec  
0        5.333900      0.746898  
1       17.253844      1.900837  
2       51.416167      5.078676  
3       84.541287      7.558295  
4      117.054094     10.384081  


In [7]:
from google.colab import files
files.download('benchmark_scalability_synthetic_converted.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
from google.colab import files
uploaded = files.upload()

Saving milestone3_cpu_randomforest.cpp to milestone3_cpu_randomforest.cpp


In [9]:
!g++ -std=c++17 -O2 -pthread -o milestone3_cpu milestone3_cpu_randomforest.cpp

In [13]:
!./milestone3_cpu

   MILESTONE 3 — Integrated Random Forest
   Yaman (Set 1) + Faraz (Set 2) + Zuhaa (Set 3)
   Hardware threads available: 2

[CPU-only benchmark export]
  [Loaded] Iris from: Iris.csv
  [Forest] Tree 1/10  nodes=5  time=0.1 ms
  [Forest] Tree 2/10  nodes=5  time=0.1 ms
  [Forest] Tree 3/10  nodes=7  time=0.1 ms
  [Forest] Tree 4/10  nodes=7  time=0.1 ms
  [Forest] Tree 5/10  nodes=5  time=0.1 ms
  [Forest] Tree 6/10  nodes=5  time=0.1 ms
  [Forest] Tree 7/10  nodes=7  time=0.1 ms
  [Forest] Tree 8/10  nodes=5  time=0.1 ms
  [Forest] Tree 9/10  nodes=5  time=0.1 ms
  [Forest] Tree 10/10  nodes=9  time=0.1 ms
  [Loaded] Letter from: letter-recognition.csv
  [Forest] Tree 1/10  nodes=291  time=80.3 ms
  [Forest] Tree 2/10  nodes=339  time=77.2 ms
  [Forest] Tree 3/10  nodes=321  time=77.4 ms
  [Forest] Tree 4/10  nodes=299  time=72.5 ms
  [Forest] Tree 5/10  nodes=339  time=73.6 ms
  [Forest] Tree 6/10  nodes=295  time=86.2 ms
  [Forest] Tree 7/10  nodes=295  time=79.5 ms
  [Forest] Tree 

In [14]:
# Cell 2: Download the CPU results
from google.colab import files
files.download('benchmark_forest_train_m3_cpu.csv')
print("✅ Downloaded benchmark_forest_train_m3_cpu.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Downloaded benchmark_forest_train_m3_cpu.csv
